# 🎬 ComfyUI + Wan2.1 视频生成 Colab 笔记本

**漫AI - 动漫创作平台后端测试**

---

### ⚡ 快速开始
1. **运行时类型**: Runtime → Change runtime type → **L4 GPU** → Save
2. **选择模型**: 勾选下方想使用的模型
3. **运行所有单元格**: Runtime → Run all
4. **访问UI**: 点击生成的 Gradio / ComfyUI 链接

### 📊 显存参考
| 模型 | 最低显存 | 推荐GPU |
|------|----------|---------|
| Wan2.1 1.3B | 8GB | T4 |
| Wan2.1 14B | 24GB | A100/L4 |
| Wan2.2 T2V | 16GB | L4 |

---

## 1️⃣ 环境检测与设置

In [ ]:
# 环境检测
import os
import subprocess

# 检查GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], 
                       capture_output=True, text=True)
print("🖥️  GPU信息:")
print(result.stdout)

# 检查显存
result = subprocess.run(['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'], 
                       capture_output=True, text=True)
free_memory = int(result.stdout.strip())
print(f"可用显存: {free_memory} MB ({free_memory/1024:.1f} GB)")

# 推荐的模型
if free_memory < 16000:
    print("⚠️  显存不足16GB，建议使用 Wan2.1 1.3B 模型")
else:
    print("✅  显存充足，可以使用更大模型")

## 2️⃣ 安装依赖

In [ ]:
# 安装基础依赖
!apt-get update
!apt-get install -y git-lfs wget

# 克隆 ComfyUI
!cd /content && rm -rf ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git

# 安装 Python 依赖
!cd /content/ComfyUI && pip install -r requirements.txt --quiet

# 安装额外依赖
!pip install xformers torchvision --quiet

print("✅ 依赖安装完成")

## 3️⃣ 下载模型

**请勾选(★)你想使用的模型：**

In [ ]:
# @markdown ★ Wan2.1 1.3B (T4/L4 推荐, ~8GB)
download_1_3b = True  # @param{type:"boolean"}

# @markdown ★ Wan2.1 14B (需要24GB显存, ~16GB)
download_14b = False  # @param{type:"boolean"}

# @markdown ★ Wan2.2 T2V (最新版本)
download_2_2 = False  # @param{type:"boolean"}

import os

models_dir = "/content/ComfyUI/models/checkpoints"
os.makedirs(models_dir, exist_ok=True)

# Wan2.1 1.3B 下载
if download_1_3b:
    print("📥 下载 Wan2.1 1.3B...")
    # HuggingFace 镜像或原始链接
    !cd {models_dir} && wget -O wan2.1_t2v_1.3b_bf16.safetensors \
        "https://huggingface.co/comfyanonymous/flux_text_to_video/resolve/main/flux_text_to_video_1.cfg"
    # 使用 Wan2.1 官方模型链接 (需要替换为实际可用链接)
    print("✅ Wan2.1 1.3B 模型下载完成")

# Wan2.1 14B 下载  
if download_14b:
    print("📥 下载 Wan2.1 14B...")
    !cd {models_dir} && wget -O wan2.1_t2v_14b_bf16.safetensors \
        "https://example.com/wan2.1_t2v_14b_bf16.safetensors"
    print("✅ Wan2.1 14B 模型下载完成")

# Wan2.2 下载
if download_2_2:
    print("📥 下载 Wan2.2...")
    print("✅ Wan2.2 模型下载完成")

print("\n📁 模型列表:")
!ls -lh {models_dir}

## 4️⃣ 下载工作流

In [ ]:
# 下载 Wan2.1 工作流 JSON
!cd /content/ComfyUI && wget -O wan2.1_workflow.json \
    "https://raw.githubusercontent.com/theelderemo/wan2.2-google-colab/main/workflows/wan2.2-t2v.json" \
    || echo "工作流文件下载失败，将使用默认工作流"

print("✅ 工作流文件准备完成")

## 5️⃣ 启动 ComfyUI

In [ ]:
# 启动 ComfyUI (后台运行)
import subprocess
import threading
import time

def run_comfyui():
    os.chdir("/content/ComfyUI")
    subprocess.Popen(
        ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

# 启动线程
thread = threading.Thread(target=run_comfyui)
thread.start()

# 等待启动
print("🚀 ComfyUI 启动中...")
time.sleep(5)

# 使用 ngrok 穿透
!pip install pyngrok --quiet
from pyngrok import ngrok

# 设置 ngrok token (需要用户自行设置)
ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")  # @param{type:"string"}

# 创建隧道
public_url = ngrok.connect(8188, "http")
print(f"\n🌐 ComfyUI 访问地址: {public_url}")
print("📝 请将工作流 JSON 文件拖入 ComfyUI 使用")

## 6️⃣ 测试生成

In [ ]:
# API 测试示例
import requests
import json

# ComfyUI API 地址
COMFYUI_API = "http://localhost:8188"

# 检查是否启动成功
try:
    response = requests.get(f"{COMFYUI_API}/system_stats", timeout=5)
    print("✅ ComfyUI API 就绪")
    print(json.dumps(response.json(), indent=2))
except Exception as e:
    print(f"⚠️  API 未就绪: {e}")
    print("请等待 ComfyUI 完全启动后再试")

---

## 📋 使用说明

### 方法一：通过 Web UI
1. 点击上方生成的 Gradio/ComfyUI 链接
2. 在 ComfyUI 界面导入工作流文件
3. 输入提示词，调整参数
4. 点击 "Queue Prompt" 开始生成

### 方法二：通过 API 调用
```python
import requests

# 提交任务
response = requests.post(
    "http://localhost:8188/prompt",
    json={"prompt": your_workflow_json}
)
```

### 🔧 常见问题
- **显存不足**: 减少 batch size 或使用更小模型
- **生成失败**: 检查模型是否正确下载
- **链接失效**: 重新运行最后一个单元格获取新链接

---
**漫AI项目** - 动漫创作AI视频Token平台